# OptimusKG — Biomedical Knowledge Graph Ingestion into Neo4j

Author: Eliézer Zarpelão

This notebook loads the **OptimusKG** multimodal biomedical knowledge graph into a **Neo4j** graph database, making it queryable through Cypher for downstream analytics, drug-disease discovery, and network-based research.

The `optimuskg` Python package provides zero-configuration access to the dataset directly from its public data lake — no manual download required.

> **Data source:** Vittor, L. et al. *OptimusKG: Unifying biomedical knowledge in a modern multimodal graph.* In Review (2026).

---

## Graph Schema

OptimusKG integrates ten biomedical entity types from curated public databases (OMIM, DrugBank, Reactome, Uberon, GO, and others).

### Nodes

| Label | Short Code | Description |
|---|---|---|
| `Gene` | `GEN` | Protein-coding and non-coding genes |
| `Disease` | `DIS` | Human diseases and disorders |
| `Drug` | `DRG` | Approved and experimental compounds |
| `Phenotype` | `PHE` | Clinical and molecular phenotypes |
| `Pathway` | `PWY` | Biological pathways (Reactome) |
| `BiologicalProcess` | `BPO` | GO biological process terms |
| `MolecularFunction` | `MFN` | GO molecular function terms |
| `CellularComponent` | `CCO` | GO cellular component terms |
| `Anatomy` | `ANA` | Anatomical structures (Uberon) |
| `Exposure` | `EXP` | Environmental and chemical exposures |

### Relationships

Edges are typed (e.g., `associated_with`, `treats`, `part_of`, `expressed_in`) and connect entities across all node types. Each edge carries provenance metadata indicating its source database(s).

## 1. Prerequisites

**Neo4j Desktop** — download and start a local Neo4j instance before running this notebook:
[https://neo4j.com/download/](https://neo4j.com/download/)

Install the required Python packages:
- `neo4j-rust-ext` — Neo4j Python driver with Rust backend for improved throughput
- `optimuskg` — zero-configuration access to the OptimusKG dataset from its public data lake

In [ ]:
%pip install neo4j-rust-ext optimuskg

In [ ]:
# Install Rust - Run the command below on terminal
# curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh

## 2. Neo4j Connection

Configure the connection to your Neo4j instance.

In [ ]:
NEO4J_URI="neo4j://127.0.0.1:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="optimuskg@123456"
NEO4J_DATABASE="neo4j"

CHUNK_SIZE_NODE = 10000
CHUNK_SIZE_REL = 5000
MAX_WORKERS = 5
MAX_RETRIES = 10

## 3. Imports & Connectivity Check

In [ ]:
import optimuskg
import json
import neo4j
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
with neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
   driver.verify_connectivity()
   print("Connection established.")

## 4. Graph Constraints

Create `NODE KEY` constraints for all ten entity types. These serve two purposes:

1. **Uniqueness** — prevent duplicate nodes if the ingestion is re-run
2. **Performance** — every constraint automatically creates a backing index, making `MERGE` and `MATCH` lookups fast at the scale of OptimusKG (~190 k nodes, ~21 M edges).

In [ ]:
constraints = [
    ["Gene", "id"],
    ["Disease", "id"],
    ["BiologicalProcess", "id"],
    ["Phenotype", "id"],
    ["Drug", "id"],
    ["Anatomy", "id"],
    ["MolecularFunction", "id"],
    ["CellularComponent", "id"],
    ["Pathway", "id"],
    ["Exposure", "id"]
]
with neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
  for c in constraints:
    query_constraint = f"CREATE CONSTRAINT constraint_{c[0].lower()} IF NOT EXISTS FOR (c:{c[0]}) REQUIRE (c.{c[1]}) IS NODE KEY"
    driver.execute_query(query_constraint, database_=NEO4J_DATABASE)
    print(query_constraint)

  result = driver.execute_query("SHOW CONSTRAINTS",
        database_=NEO4J_DATABASE,
        result_transformer_=neo4j.Result.to_df
      )
  print(result)

## 5. Load OptimusKG

The `load_graph()` call fetches two Polars DataFrames from the public data lake (downloaded once, then cached locally):

| DataFrame | Columns | Description |
|---|---|---|
| `nodes` | `id`, `label`, `properties` | All biomedical entities |
| `edges` | `from`, `to`, `label`, `relation`, `properties` | All typed relationships |

`label` on edges encodes the source and target types as `SRC-TGT` (e.g., `GEN-DIS` for Gene → Disease edges).

In [ ]:
# Load nodes and edges 
nodes, edges = optimuskg.load_graph()

## 6. Helper Functions

Shared utilities used by both node and edge ingestion pipelines:

- **`label_mapping`** — maps short label codes (e.g. `GEN`, `DIS`) to full Neo4j node labels (e.g. `Gene`, `Disease`)
- **`sanitize_properties`** — normalises nested JSON structures so all property values are Neo4j-compatible primitives (strings, numbers, booleans, or flat lists)
- **`log`** — timestamped print with elapsed pipeline time; re-run its cell to reset the timer
- **`ingest_node_chunk`** / **`ingest_edge_chunk`** — ingest a single chunk into Neo4j, retrying on `TransientError` (lock/deadlock) with exponential backoff up to `MAX_RETRIES` attempts

In [ ]:
# Map short label codes to descriptive Neo4j node labels
label_mapping = {
    "GEN": "Gene",
    "DIS": "Disease",
    "BPO": "BiologicalProcess",
    "PHE": "Phenotype",
    "DRG": "Drug",
    "ANA": "Anatomy",
    "MFN": "MolecularFunction",
    "CCO": "CellularComponent",
    "PWY": "Pathway",
    "EXP": "Exposure"
}

In [ ]:
def sanitize_properties(properties):
    """
    Recursively sanitize a dictionary to ensure all values are Neo4j-compatible
    primitives (string, number, boolean, list of primitives).
    Dicts and lists containing dicts are converted to JSON strings.
    """
    sanitized = {}
    for key, value in properties.items():
        if isinstance(value, dict):
            sanitized[key] = json.dumps(value)
        elif isinstance(value, list):
            sanitized[key] = [json.dumps(item) if isinstance(item, dict) else str(item) for item in value]
        elif isinstance(value, (str, int, float, bool)) or value is None:
            sanitized[key] = value
        else:
            sanitized[key] = str(value)
    return sanitized

The cell below defines `log`, `ingest_node_chunk`, and `ingest_edge_chunk`. Re-run it to reset the pipeline timer.

In [ ]:
import time
from datetime import datetime
from neo4j.exceptions import TransientError

_pipeline_start = time.perf_counter()

def log(msg):
    """Print with absolute timestamp and elapsed time since pipeline start."""
    elapsed = time.perf_counter() - _pipeline_start
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts} | +{elapsed:7.2f}s] {msg}")

def ingest_node_chunk(driver, new_label, nodes_to_load, chunk_idx):
    """Ingest a single chunk of nodes; retry on transient lock/deadlock errors."""
    query = f"""
    UNWIND $nodes_data AS node_info
    CREATE (n:{new_label} {{id: node_info.id}})
    SET n += node_info.properties
    """
    for attempt in range(1, MAX_RETRIES + 1):
        t0 = time.perf_counter()
        try:
            with driver.session(database=NEO4J_DATABASE) as session:
                session.run(query, nodes_data=nodes_to_load)
            log(f"  [{new_label}] chunk {chunk_idx} done in {time.perf_counter() - t0:.2f}s ({len(nodes_to_load)} nodes)")
            return
        except TransientError:
            if attempt < MAX_RETRIES:
                wait = 2 ** attempt
                log(f"  [{new_label}] chunk {chunk_idx} lock error (attempt {attempt}/{MAX_RETRIES}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

def ingest_edge_chunk(driver, new_from_label, new_to_label, relation_type, edges_to_load, chunk_idx):
    """Ingest a single chunk of edges; retry on transient lock/deadlock errors."""
    query = f"""
    UNWIND $edges_data AS edge_info
    MATCH (source_node:{new_from_label} {{id: edge_info.from}})
    MATCH (target_node:{new_to_label} {{id: edge_info.to}})
    CREATE (source_node)-[r:`{relation_type}`]->(target_node)
    SET r += edge_info.properties
    """
    for attempt in range(1, MAX_RETRIES + 1):
        t0 = time.perf_counter()
        try:
            with driver.session(database=NEO4J_DATABASE) as session:
                session.run(query, edges_data=edges_to_load)
            log(f"  [{relation_type}] chunk {chunk_idx} done in {time.perf_counter() - t0:.2f}s ({len(edges_to_load)} edges)")
            return
        except TransientError:
            if attempt < MAX_RETRIES:
                wait = 2 ** attempt
                log(f"  [{relation_type}] chunk {chunk_idx} lock error (attempt {attempt}/{MAX_RETRIES}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## 7. Node Ingestion

All label types are ingested concurrently using `ThreadPoolExecutor`. Each worker handles one chunk for one label, opening its own Neo4j session (the driver is thread-safe and shared across workers).

In [ ]:
with neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    log("Connected to Neo4j — loading nodes.")

    tasks = []
    log(f"Found {len(label_mapping)} label types. Starting DataFrame filtering...")

    for original_label, new_label in label_mapping.items():
        t0 = time.perf_counter()
        log(f"  Filtering '{original_label}'...")
        current_label_nodes = nodes.filter(nodes["label"] == original_label)
        log(f"  Filtered '{original_label}' → {len(current_label_nodes):,} rows in {time.perf_counter() - t0:.2f}s")

        if current_label_nodes.is_empty():
            log(f"  No nodes found for label '{original_label}'.")
            continue

        for i in range(0, len(current_label_nodes), CHUNK_SIZE_NODE):
            chunk = current_label_nodes[i : i + CHUNK_SIZE_NODE]
            nodes_to_load = []
            for row_dict in chunk.to_dicts():
                try:
                    props = sanitize_properties(json.loads(row_dict['properties']))
                except json.JSONDecodeError:
                    log(f"  Err: json.JSONDecodeError for node {row_dict['id']}")
                    continue
                nodes_to_load.append({"id": row_dict['id'], "properties": props})
            if nodes_to_load:
                tasks.append((new_label, nodes_to_load, i // CHUNK_SIZE_NODE))

    log(f"DataFrame filtering complete. Submitting {len(tasks)} chunks across {len(label_mapping)} label types...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(ingest_node_chunk, driver, label, data, idx): (label, idx)
            for label, data, idx in tasks
        }
        for future in as_completed(futures):
            label, idx = futures[future]
            try:
                future.result()
            except Exception as e:
                log(f"  Error in [{label}] chunk {idx}: {e}")

log("Node ingestion complete.")

## 8. Edge Ingestion

Edges are grouped by `(label, relation)` pair — relationship types must be embedded as literals in Cypher, so each group gets its own query. All chunks across all groups are submitted concurrently via `ThreadPoolExecutor`.

Because concurrent writes can produce deadlocks, each worker retries on `TransientError` (Neo4j's lock/deadlock signal) up to `MAX_RETRIES` times with exponential backoff before propagating the failure.

In [ ]:
with neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    log("Connected to Neo4j — loading edges.")

    tasks = []
    unique_edge_relation_tuples = edges.select(['label', 'relation']).unique().to_dicts()
    log(f"Found {len(unique_edge_relation_tuples)} unique edge types. Starting DataFrame filtering...")

    for group_info in unique_edge_relation_tuples:
        edge_label = group_info['label']
        relation_type = group_info['relation']

        try:
            src_abbr, tgt_abbr = edge_label.split('-')
            new_from_label = label_mapping.get(src_abbr, src_abbr)
            new_to_label = label_mapping.get(tgt_abbr, tgt_abbr)
        except ValueError:
            log(f"  Error: invalid edge label '{edge_label}'. Skipping.")
            continue

        t0 = time.perf_counter()
        log(f"  Filtering '{edge_label}' ({relation_type})...")
        current_group_edges = edges.filter(
            (edges["label"] == edge_label) & (edges["relation"] == relation_type)
        )
        log(f"  Filtered '{edge_label}' → {len(current_group_edges):,} rows in {time.perf_counter() - t0:.2f}s")

        if current_group_edges.is_empty():
            log(f"  No edges found for label '{edge_label}' with relation '{relation_type}'.")
            continue

        for i in range(0, len(current_group_edges), CHUNK_SIZE_REL):
            chunk = current_group_edges[i : i + CHUNK_SIZE_REL]
            edges_to_load = []
            for row_dict in chunk.to_dicts():
                try:
                    props = sanitize_properties(json.loads(row_dict['properties']))
                except json.JSONDecodeError:
                    log(f"  Err: json.JSONDecodeError for edge from={row_dict['from']} to={row_dict['to']}")
                    continue
                edges_to_load.append({"from": row_dict['from'], "to": row_dict['to'], "properties": props})
            if edges_to_load:
                tasks.append((new_from_label, new_to_label, relation_type, edges_to_load, i // CHUNK_SIZE_REL))

    log(f"DataFrame filtering complete. Submitting {len(tasks)} chunks across {len(unique_edge_relation_tuples)} edge types...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(ingest_edge_chunk, driver, from_l, to_l, rel, data, idx): (rel, idx)
            for from_l, to_l, rel, data, idx in tasks
        }
        for future in as_completed(futures):
            rel, idx = futures[future]
            try:
                future.result()
            except Exception as e:
                log(f"  Error in [{rel}] chunk {idx}: {e}")

log("Edge ingestion complete.")

# Open Issues

https://github.com/mims-harvard/OptimusKG/issues/184